### Testing Reranker Model

In [1]:
import os
from pathlib import Path
import gc

# Workaround for OpenMP duplicate runtime crash on Windows.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch
from transformers import AutoModel, AutoTokenizer, BertTokenizer, BertModel

model_dir = Path(r"d:\Projects\BotMaker\OpenScholar_Retriever")
assert model_dir.exists(), f"Model directory not found: {model_dir}"

def pick_device():
    if torch.cuda.is_available():
        try:
            _ = torch.empty((1,), device="cuda")
            return torch.device("cuda")
        except Exception as exc:
            print(f"CUDA init failed, falling back to CPU: {exc}")
    return torch.device("cpu")

device = pick_device()
torch.set_num_threads(1)

c:\Users\cszmli\AppData\Local\miniconda3\envs\huggingface-download\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load tokenizer and encoder from the local OpenScholar_Retriever folder.
tokenizer = AutoTokenizer.from_pretrained(str(model_dir), local_files_only=True)
model = AutoModel.from_pretrained(str(model_dir), local_files_only=True)
model.eval()

try:
    model.to(device)
    if device.type == "cuda":
        model = model.half()
except Exception as exc:
    print(f"Model to {device} failed, using CPU: {exc}")
    device = torch.device("cpu")
    model.to(device)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 38986.41it/s]
[transformers] BertModel LOAD REPORT from: d:\Projects\BotMaker\OpenScholar_Retriever
Key                 | Status  | 
--------------------+---------+-
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
passages = [
    "Cloud-based active content collaboration platform using multimedia processing In this article, a new content-centric collaboration platform called active content collaboration platform which supports automated content-centric collaboration on cloud system is presented. It supports event-driven automatic collaboration by specifying each active work based on active work description language and automating the execution of collaborative task flow composed of active works. It supports a modularized and extensible architecture by making its components in separate modules. Also, it provides a scalable high-performance architecture by supporting multi-level active work processing in active work execution engine, and allocating virtual machines for the computation intensive high-level action such as volume rendering through auto-scalable allocation on cloud system. For the experiment and evaluation, we shall show the results of implementing the collaborative medical application on our system where computation intensive application like a volume rendering is used for MRI analysis. Introduction Nowadays, collaboration organization becomes more complicated and grows bigger. Therefore, it requires huge collaboration platform through networks. Modern softwares for content-centric collaborations support web applications for business tasks based on traditional software systems such as enterprise content management (ECM) [1] or repository softwares. Those content-centric collaboration platforms reduce network traffic for collaboration content transfer. Previous collaboration platforms were developed to provide private, isolated, and unique methods used for specific organization [2]. It means that those softwares should be customized for each organization for providing collaboration services, and hence it is hard to retain them flexibly. Furthermore, collaboration between organizations with different systems requires additional cost, and makes security problems. This limitation of"
]

batch = tokenizer(
    passages,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt",
).to(device)
outputs = model(**batch)

In [3]:
query = "What methods improve retrieval for scientific literature search?"
passages = [
    "Cloud-based active content collaboration platform using multimedia processing In this article, a new content-centric collaboration platform called active content collaboration platform which supports automated content-centric collaboration on cloud system is presented. It supports event-driven automatic collaboration by specifying each active work based on active work description language and automating the execution of collaborative task flow composed of active works. It supports a modularized and extensible architecture by making its components in separate modules. Also, it provides a scalable high-performance architecture by supporting multi-level active work processing in active work execution engine, and allocating virtual machines for the computation intensive high-level action such as volume rendering through auto-scalable allocation on cloud system. For the experiment and evaluation, we shall show the results of implementing the collaborative medical application on our system where computation intensive application like a volume rendering is used for MRI analysis. Introduction Nowadays, collaboration organization becomes more complicated and grows bigger. Therefore, it requires huge collaboration platform through networks. Modern softwares for content-centric collaborations support web applications for business tasks based on traditional software systems such as enterprise content management (ECM) [1] or repository softwares. Those content-centric collaboration platforms reduce network traffic for collaboration content transfer. Previous collaboration platforms were developed to provide private, isolated, and unique methods used for specific organization [2]. It means that those softwares should be customized for each organization for providing collaboration services, and hence it is hard to retain them flexibly. Furthermore, collaboration between organizations with different systems requires additional cost, and makes security problems. This limitation of"
]

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

@torch.inference_mode()
def encode(texts):
    batch = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt",
    ).to(device)
    outputs = model(**batch)
    embeddings = mean_pooling(outputs.last_hidden_state, batch["attention_mask"])
    # embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
    return embeddings

if device.type == "cuda":
    torch.cuda.empty_cache()
    gc.collect()

query_emb = encode([query])
passage_embs = encode(passages)
scores = (query_emb @ passage_embs.T).squeeze(0)
ranked = sorted(zip(passages, scores.tolist()), key=lambda x: x[1], reverse=True)

print(f"Device: {device}")
print("\nQuery:")
print(query)
print("\nRanked passages:")
for i, (passage, score) in enumerate(ranked, 1):
    print(f"{i}. {score:.4f} | {passage}")

Device: cuda

Query:
What methods improve retrieval for scientific literature search?

Ranked passages:
1. 0.6489 | Cloud-based active content collaboration platform using multimedia processing In this article, a new content-centric collaboration platform called active content collaboration platform which supports automated content-centric collaboration on cloud system is presented. It supports event-driven automatic collaboration by specifying each active work based on active work description language and automating the execution of collaborative task flow composed of active works. It supports a modularized and extensible architecture by making its components in separate modules. Also, it provides a scalable high-performance architecture by supporting multi-level active work processing in active work execution engine, and allocating virtual machines for the computation intensive high-level action such as volume rendering through auto-scalable allocation on cloud system. For the experi

In [11]:
passage_embs[0].cpu().numpy()

array([ 1.02312379e-02,  6.19368348e-03, -3.12818736e-02,  8.88050068e-03,
       -1.26975030e-03,  2.70267371e-02, -4.77650836e-02, -2.92646252e-02,
        8.44679847e-02, -5.20361029e-03, -6.07697815e-02, -2.14275569e-02,
        2.26130169e-02,  1.49273679e-01,  8.24609865e-03,  4.97357026e-02,
        1.75699331e-02, -1.73045211e-02, -1.17338032e-01, -2.53000334e-02,
       -5.04344031e-02, -8.58182013e-02,  1.90996397e-02,  2.08049621e-02,
       -7.73609728e-02, -1.24805942e-02, -5.72732836e-02,  3.55183147e-04,
        2.96207462e-02, -2.38417909e-02, -5.65227866e-02, -3.67978737e-02,
        3.88785005e-02, -7.08333403e-02, -7.29994848e-03,  3.91499028e-02,
       -5.38203567e-02, -3.53747308e-02,  3.51867788e-02, -2.07293853e-02,
        9.94831026e-02, -9.18635279e-02,  3.32608968e-02, -7.96443075e-02,
       -5.64805381e-02, -5.71778789e-02, -3.42300572e-02,  7.04211742e-02,
        1.93215255e-02,  2.82958299e-02, -8.80376175e-02, -8.96323100e-02,
       -8.57719406e-02,  

In [12]:
# passage_embs[0] 保存为 numpy 数组
import numpy as np
np.save("passage_emb_0.npy", passage_embs[0].cpu().numpy())

### Read OpenScholar-DataStore-V3\passages\raw_passages-0-of-16.jsonl content

In [1]:
from pathlib import Path
import json

# OpenScholar-DataStore-V3\passages\raw_passages-0-of-16.jsonl 提取前50条记录
data_path = Path(r"d:\Projects\BotMaker\OpenScholar-DataStore-V3\passages\raw_passages-1-of-16.jsonl")
with data_path.open("r", encoding="utf-8") as f:
    for _ in range(50):
        line = f.readline()
        if not line:
            break
        record = json.loads(line)
        print(record)

{'text': 'Cloud-based active content collaboration platform using multimedia processing In this article, a new content-centric collaboration platform called active content collaboration platform which supports automated content-centric collaboration on cloud system is presented. It supports event-driven automatic collaboration by specifying each active work based on active work description language and automating the execution of collaborative task flow composed of active works. It supports a modularized and extensible architecture by making its components in separate modules. Also, it provides a scalable high-performance architecture by supporting multi-level active work processing in active work execution engine, and allocating virtual machines for the computation intensive high-level action such as volume rendering through auto-scalable allocation on cloud system. For the experiment and evaluation, we shall show the results of implementing the collaborative medical application on ou

### Read D:\Projects\BotMaker\OpenScholar-DataStore-V3\embeddings\passages_00.pkl

In [ ]:
import os
import pickle
import pickletools
from pathlib import Path

# pkl_path = Path(r"D:\Projects\BotMaker\OpenScholar-DataStore-V3\embeddings\passages_00.pkl")
pkl_path = Path(r"C:\Projects\Biomaker_tmpdata\tmp_sentence_embeddings\raw_passages-0-of-16.first_sentences.embeddings.f16.pkl")

rows = []
with pkl_path.open("rb") as f:
    unpickler = pickle.Unpickler(f)
    while len(rows) < 2:
        try:
            rows.append(unpickler.load())
        except EOFError:
            break

print(f"Read {len(rows)} record(s) from pickle stream")
for i, row in enumerate(rows, 1):
    print(f"[{i}] type={type(row)}")
    print(row)

File size: 4,664,160,656 bytes
First top-level pickle ends at: 61,440,140 bytes
Multiple top-level objects: True
Read 2 record(s) from pickle stream
[1] type=<class 'numpy.ndarray'>
[[ 0.013695  -0.0315     0.085     ... -0.02545   -0.0521    -0.03162  ]
 [-0.02782    0.01822    0.01346   ... -0.01857   -0.04578   -0.02232  ]
 [-0.1056    -0.1044    -0.02483   ...  0.008255   0.05597   -0.0461   ]
 ...
 [-0.0292    -0.04288   -0.04883   ... -0.0684     0.03595    0.03638  ]
 [ 0.004684  -0.01572   -0.007195  ... -0.024     -0.00618   -0.0385   ]
 [-0.0002964 -0.04166   -0.003191  ... -0.001952  -0.07      -0.0124   ]]
[2] type=<class 'numpy.ndarray'>
[[ 0.02953    0.02406   -0.0197    ... -0.03455    0.00784   -0.003435 ]
 [-0.1324    -0.01369    0.03464   ... -0.003288   0.002806   0.002254 ]
 [-0.0112     0.01096    0.01283   ... -0.00935   -0.0891    -0.02208  ]
 ...
 [ 0.04178   -0.02814   -0.04712   ... -0.0568    -0.0274    -0.0544   ]
 [-0.10913    0.0003345  0.0287    ...  0.04